# Activation → PCA

Open the parquet, run the PCA, show the loadings. No project imports — pandas,
numpy, matplotlib.

Edit `SELECT`. `"*"` means every subject in that cohort; drop a cohort to
exclude it. One atlas at a time (7 / 14 / 111 parcels, different schemas).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))
ATLAS = "yeo7"
SELECT = {"camcan": ["*"]}
N_COMPONENTS = 5

# Everything in an activation file that is not a parcel.
META = ["t", "time_s", "stimulus_time_s", "good_frame", "run_idx",
        "ses", "run", "acq", "run_key"]
KEYS = ["atlas", "cohort", "task", "sub"]

paths = [p
         for cohort, subs in SELECT.items()
         for sub in subs
         for p in sorted(ROOT.glob(
             f"activation/atlas={ATLAS}/cohort={cohort}/task=*/sub={sub}/data.parquet"))]
print(f"{len(paths)} shard(s)")

# The partition keys are directory names, so put them back as columns.
act = pd.concat(
    [pd.read_parquet(p).assign(**dict(s.split("=", 1) for s in p.parts if "=" in s))
     for p in paths],
    ignore_index=True)

parcels = [c for c in act.columns if c not in META and c not in KEYS]
print(f"{len(act):,} rows x {len(parcels)} parcels, "
      f"{act['sub'].nunique()} subject(s)")
act.head(3)

In [ ]:
X = act.loc[act["good_frame"].to_numpy(bool), parcels]
X = X.dropna(axis=1, how="all").dropna(axis=0)     # empty parcels, then leftover NaN rows

# Watch the subject count. A parcel that is empty for ONE subject survives the
# column drop, and the row drop then removes that subject entirely -- so a
# shrinking subject count here means you lost people, not frames.
print(f"rows     {int(act['good_frame'].sum()):,} good -> {len(X):,}")
print(f"parcels  {len(parcels)} -> {X.shape[1]}")
print(f"subjects {act['sub'].nunique()} -> {act.loc[X.index, 'sub'].nunique()}")

M = X.to_numpy(dtype=float, copy=True)
M -= M.mean(axis=0)
sd = M.std(axis=0, ddof=1)
M /= np.where(sd == 0, 1.0, sd)                    # z-score: parcels are not on one scale

U, S, Vt = np.linalg.svd(M, full_matrices=False)
ratio = S**2 / (S**2).sum()
k = min(N_COMPONENTS, len(S))
loadings = pd.DataFrame(Vt[:k].T, index=X.columns,
                        columns=[f"PC{i+1}" for i in range(k)])

pd.DataFrame({"explained": ratio[:k].round(4),
              "cumulative": ratio[:k].cumsum().round(4)},
             index=loadings.columns)

In [ ]:
loadings.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(min(0.35 * len(loadings) + 3, 18), 0.5 * k + 2))
v = np.abs(loadings.to_numpy()).max()
im = ax.imshow(loadings.T, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v)
ax.set_yticks(range(k)); ax.set_yticklabels(loadings.columns)
if len(loadings) <= 40:                # 111 parcel names do not fit on an axis
    ax.set_xticks(range(len(loadings)))
    ax.set_xticklabels(loadings.index, rotation=90, fontsize=7)
else:
    ax.set_xlabel(f"{len(loadings)} parcels, in label-table order")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()